In [10]:
# !pip install aiohttp
import re
import asyncio
import aiohttp
import pandas as pd
from tqdm.notebook import tqdm
import os
import time

In [17]:
INPUT_CSV   = "input/unique_video_urls_p06.csv"
OUTPUT_CSV  = "output/creator_map_final_p06.csv"
PARTIAL_CSV = "output/creator_map_partial_p06.csv"

CONCURRENCY = 10    # 10 requests tegelijk; verhoog naar 20 als je geen 429s ziet
DELAY_S     = 0.3   # wachttijd per worker
TIMEOUT_S   = 15
SAVE_EVERY  = 500
RESUME      = True

HEADERS = {"User-Agent": "Mozilla/5.0"}

In [11]:
async def resolve_creator_async(session: aiohttp.ClientSession,
                                 semaphore: asyncio.Semaphore,
                                 url: str) -> dict:
    out = {
        "input_url":        url,
        "final_url":        None,
        "creator_username": None,
        "status_code":      None,
        "error":            None,
    }

    async with semaphore:
        try:
            timeout = aiohttp.ClientTimeout(total=TIMEOUT_S)

            # Stap 1: redirect volgen
            async with session.get(url, headers=HEADERS,
                                   allow_redirects=True,
                                   timeout=timeout) as r:
                out["final_url"] = str(r.url)

            # Stap 2: oEmbed met final_url
            oembed_url = f"https://www.tiktok.com/oembed?url={out['final_url']}"
            async with session.get(oembed_url, headers=HEADERS,
                                   timeout=timeout) as r2:
                out["status_code"] = r2.status
                if r2.status == 200:
                    data = await r2.json(content_type=None)
                    out["creator_username"] = data.get("author_name")
                else:
                    out["error"] = f"http_{r2.status}"

        except asyncio.TimeoutError:
            out["error"] = "timeout"
        except Exception as e:
            out["error"] = f"{e.__class__.__name__}: {e}"
        finally:
            await asyncio.sleep(DELAY_S)

    return out

In [12]:
async def run_all(urls: list[str]) -> pd.DataFrame:
    semaphore = asyncio.Semaphore(CONCURRENCY)
    results   = []
    pbar      = tqdm(total=len(urls), desc="Resolving", unit="url")

    connector = aiohttp.TCPConnector(limit=CONCURRENCY + 5, ssl=False)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [resolve_creator_async(session, semaphore, u) for u in urls]

        for coro in asyncio.as_completed(tasks):
            result = await coro
            results.append(result)
            pbar.update(1)

            ok  = sum(1 for r in results if r["creator_username"])
            pbar.set_postfix(ok=ok, pct=f"{100*ok/len(results):.0f}%")

            # Dit (huidige sessie + alle vorige sessies):
            if len(results) % SAVE_EVERY == 0:
                partial = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True) if done_df is not None else pd.DataFrame(results)
                partial.to_csv(PARTIAL_CSV, index=False)

    pbar.close()
    return pd.DataFrame(results)

In [18]:
all_urls = pd.read_csv(INPUT_CSV)["url"].tolist()

if os.path.exists(PARTIAL_CSV):
    done_df   = pd.read_csv(PARTIAL_CSV)
    done_set  = set(done_df["input_url"].dropna())
    urls_todo = [u for u in all_urls if u not in done_set]
    print(f"Al gedaan: {len(done_set)} | Nog te doen: {len(urls_todo)}")
else:
    urls_todo = all_urls
    done_df   = None
    print(f"Te doen: {len(urls_todo)}")

Al gedaan: 49411 | Nog te doen: 297587


In [ ]:
t0     = time.time()
new_df = await run_all(urls_todo)
elapsed = time.time() - t0

out = pd.concat([done_df, new_df], ignore_index=True) if done_df is not None else new_df
out.to_csv(OUTPUT_CSV, index=False)

print(f"Klaar in {elapsed/60:.1f} minuten")
print(f"Creator gevonden: {out['creator_username'].notna().sum()}/{len(out)}")

Resolving:   0%|          | 0/297587 [00:00<?, ?url/s]

In [14]:
print(new_df['status_code'].value_counts())
print()
print(new_df['error'].value_counts().head(10))
print()
# Hoeveel hebben helemaal geen result (None)?
print("Lege creator_username:")
print(new_df[new_df['creator_username'].isna()][['input_url','status_code','error']].head(10))

NameError: name 'new_df' is not defined

In [14]:
# Herrun alleen de timeouts
timeout_urls = out[out['error'] == 'timeout']['input_url'].tolist()
print(f"Te hernemen: {len(timeout_urls)}")

# done_df instellen zodat RESUME werkt
done_df = out[out['error'] != 'timeout'].copy()

t0      = time.time()
retry_df = await run_all(timeout_urls)
elapsed  = time.time() - t0

# Samenvoegen
out_final = pd.concat([done_df, retry_df], ignore_index=True)
out_final.to_csv(OUTPUT_CSV, index=False)

found = out_final['creator_username'].notna().sum()
print(f"Klaar in {elapsed/60:.1f} min")
print(f"Creator gevonden: {found}/{len(out_final)} ({100*found/len(out_final):.1f}%)")

Te hernemen: 4970


Resolving:   0%|          | 0/4970 [00:00<?, ?url/s]

CancelledError: 

In [10]:
#Rerun 403
# Laad het huidige output bestand
out = pd.read_csv(OUTPUT_CSV)

# Herrun alleen de 403s
retry_urls = out[out['error'] == 'http_403']['input_url'].tolist()
print(f"Te hernemen: {len(retry_urls)}")

# Alles behalve de 403s behouden
done_df = out[out['error'] != 'http_403'].copy()

t0       = time.time()
retry_df = await run_all(retry_urls)
elapsed  = time.time() - t0

# Samenvoegen en opslaan
out_final = pd.concat([done_df, retry_df], ignore_index=True)
out_final.to_csv(OUTPUT_CSV, index=False)

found = out_final['creator_username'].notna().sum()
print(f"Klaar in {elapsed/60:.1f} min")
print(f"Creator gevonden: {found}/{len(out_final)} ({100*found/len(out_final):.1f}%)")
print(out_final['error'].value_counts().head(10))

Te hernemen: 732


Resolving:   0%|          | 0/732 [00:00<?, ?url/s]

Klaar in 3.0 min
Creator gevonden: 117832/131440 (89.6%)
error
http_400                                                                                                                  13587
ClientConnectorDNSError: Cannot connect to host www.tiktokv.com:443 ssl:default [Timeout while contacting DNS servers]       15
timeout                                                                                                                       5
Name: count, dtype: int64


In [15]:
# Alles ophalen uit de partial CSV
out_final = pd.read_csv(PARTIAL_CSV)

found = out_final['creator_username'].notna().sum()
print(f"Totaal rijen:     {len(out_final)}")
print(f"Creator gevonden: {found} ({100*found/len(out_final):.1f}%)")
print()
print(out_final['status_code'].value_counts())
print()
print(out_final['error'].value_counts().head(10))

Totaal rijen:     48911
Creator gevonden: 41867 (85.6%)

status_code
200.0    41867
400.0     7044
Name: count, dtype: int64

error
http_400    7044
Name: count, dtype: int64


In [9]:
out_final.to_csv(OUTPUT_CSV, index=False)
print("Opgeslagen!")

Opgeslagen!
